In [1]:
# ===== Summarize prompt robustness results =====
# Reads NewPrompt*.csv files and calculates A/B/C counts and percentages
# grouped by model and prompt_version

import os
import re
import glob
import pandas as pd

BASE_DIR = "."
PATTERN = "NewPrompt*.csv"

OUT_DIR = os.path.join(BASE_DIR, "out")
os.makedirs(OUT_DIR, exist_ok=True)


def extract_choice_letter(ans):
    """Extract A/B/C from answers like 'A - reason' or 'Answer: B - reason'"""

    if not isinstance(ans, str) or not ans.strip():
        return None

    s = re.sub(r'^\s*Answer:\s*', '', ans, flags=re.IGNORECASE).strip()

    m = re.match(r'^\s*([ABC])\b', s, flags=re.IGNORECASE)

    if not m:
        m = re.search(r'\b([ABC])\b', s[:80], flags=re.IGNORECASE)

    return m.group(1).upper() if m else None


# ---- Find files ----

files = sorted(glob.glob(os.path.join(BASE_DIR, PATTERN)))

if not files:
    raise SystemExit(f"No files matched {PATTERN}")


rows = []

for file in files:

    df = pd.read_csv(file)

    if "answer" not in df.columns:
        raise ValueError(f"{file} missing 'answer' column")

    if "prompt_version" not in df.columns:
        raise ValueError(f"{file} missing 'prompt_version' column")

    model_name = df["model"].iloc[0]

    df["choice"] = df["answer"].map(extract_choice_letter)

    valid = df.dropna(subset=["choice"])

    for prompt, group in valid.groupby("prompt_version"):

        counts = group["choice"].value_counts()

        A = int(counts.get("A", 0))
        B = int(counts.get("B", 0))
        C = int(counts.get("C", 0))

        total = int(group.shape[0])

        rows.append({
            "model": model_name,
            "prompt": prompt,
            "A": A,
            "B": B,
            "C": C,
            "total": total,
            "pct_A": round(100*A/total, 2) if total else 0,
            "pct_B": round(100*B/total, 2) if total else 0,
            "pct_C": round(100*C/total, 2) if total else 0
        })


summary = pd.DataFrame(rows).sort_values(["model","prompt"]).reset_index(drop=True)


# ---- Save CSV ----

csv_path = os.path.join(OUT_DIR, "prompt_choice_counts.csv")

summary.to_csv(csv_path, index=False)

print(f"[OK] Wrote {csv_path}")


print("\nPrompt robustness summary:\n")

print(summary.to_string(index=False))


# ---- LaTeX table for paper ----

try:

    from textwrap import dedent

    latex_df = summary.copy()

    latex_df = latex_df.rename(columns={
        "pct_A": r"\makecell{\%\\A}",
        "pct_B": r"\makecell{\%\\B}",
        "pct_C": r"\makecell{\%\\C}"
    })

    tex_path = os.path.join(OUT_DIR, "prompt_choice_counts_table.tex")

    with open(tex_path, "w", encoding="utf-8") as f:

        f.write(dedent(r"""
        % Requires: \usepackage{booktabs}
        % Optional: \usepackage{makecell}
        """).lstrip())

        f.write(latex_df.to_latex(index=False, float_format="%.2f", escape=False))

    print(f"[OK] Wrote {tex_path}")

except Exception as e:

    print(f"[WARN] Could not write LaTeX table: {e}")

[OK] Wrote .\out\prompt_choice_counts.csv

Prompt robustness summary:

                            model         prompt  A  B  C  total  pct_A  pct_B  pct_C
            google/gemma-3-27b-it natural_advice 15 23 82    120  12.50  19.17  68.33
meta-llama/llama-3.1-70b-instruct natural_advice 14 47 59    120  11.67  39.17  49.17
                    openai/gpt-4o natural_advice  5 17 98    120   4.17  14.17  81.67
                 qwen/qwen2.5-32b natural_advice 12 29 79    120  10.00  24.17  65.83
[OK] Wrote .\out\prompt_choice_counts_table.tex
